** 代码与文字尚未正确——请忽略！！！ **

## 错误测量（Erroneous Measurements）

如果你用的传感器从不故障、从不给出虚假测量，那你很幸运——可以直接跳到下一节。现实中传感器并不完美：杂散电压影响信号，鸟会飞过距离传感器前方，计算机视觉传感器会把阴影误认成行人，诸如此类。

整本书都在讨论这个话题。我觉得专门讲雷达多目标跟踪的书特别有用。这里我只谈几个要点，用很少的理论和代码就能显著改善你的滤波器。

我们已经讨论过*似然（likelihood）*函数。回顾贝叶斯法则（Bayes' rule）：

$$\text{posterior} = \frac{\text{likelihood}\times\text{prior}}{\text{evidence}}$$

其中似然定义为

$$\mathcal L = p(\mathbf z \mid \mathbf x)$$

即给定先验 $\mathbf x$ 时测量值（measurement）的似然（概率）。这暗示了一个简单的*门控（gating）*函数。这里的*门控*指根据某些准则接受或拒绝测量的函数。我们假设测量噪声（measurement noise）是高斯（Gaussian）的。回忆高斯（Gaussians）章节：99.7% 的值落在均值 3 个标准差以内。若测量 $\mathbf z > 3 \sigma$，可以认为极不可能而丢弃。很简单。

实践中你或许不该这么做。我们把传感器*建模*为高斯，但实际未必如此：尾部可能更厚，分布可能略不同（如 Student t 分布等）。例如 NASA 在猎户座（Orion）任务中为适应传感器真实行为，使用 $5\sigma$ 到 $6\sigma$。你的预算大概不如 NASA，但风险可能也没那么大。无法解析地确定正确数值——你需要用自己的数据做实验，看对你的应用什么才是合理取值。

In [ ]:
from math import sqrt

def gated_fusion(pos_data, vel_data, wheel_std, ps_std, gate=3.):
    kf = KalmanFilter(dim_x=2, dim_z=1)
    kf.F = array([[1., 1.], [0., 1.]])
    kf.H = array([[1., 0.], [1., 0.]])
    kf.x = array([[0.], [1.]])
    kf.P *= 100

    xs, ts = [],  []
    
    # copy data for plotting
    zs_wheel = np.array(vel_data)
    zs_ps = np.array(pos_data)
                     
    last_t = 0
    while len(pos_data) > 0 and len(vel_data) > 0:
        if pos_data[0][0] < vel_data[0][0]:
            t, z = pos_data.pop(0)
            dt = t - last_t
            last_t = t
            p_index = 0
            
            kf.H = np.array([[1., 0.]])
            kf.R[0,0] = ps_std**2
            si
        else:
            t, z = vel_data.pop(0)
            dt = t - last_t
            last_t = t
            p_index = 1
            
            kf.H = np.array([[0., 1.]])
            kf.R[0,0] = wheel_std**2

        kf.F[0,1] = dt
        kf.Q = Q_discrete_white_noise(2, dt=dt, var=.02)
        kf.predict()
        std = sqrt(kf.P[p_index, p_index])
        y = abs(kf.residual_of(z)) 
        if  y <= std * gate:
            kf.update(np.array([z]))
        else:
            print('skip', p_index, kf.x.T, kf.P.diagonal(), "%.3f" % z, y)


        xs.append(kf.x.T[0])
        ts.append(t)

    plot_fusion(xs, ts, zs_ps, zs_wheel)
    
random.seed(1123)
pos_data, vel_data = gen_sensor_data(25, 1.5, .3)
gated_fusion(pos_data, vel_data, 1.5, 3.0, gate=4.);
